In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

root = Path.cwd().parent

df = pd.read_csv(root / "data/processed/nrc/nrc_reactor_daily.csv")

df.head()

,date,region,unit_name,power_pct,down_raw,reason_comment,change_raw,scrams_raw
0,1999-01-01,4,Arkansas Nuclear 1,0.0,1998-12-28,Manual Reactor Trip. See event 35199. Design...,NaN,NaN
1,1999-01-02,4,Arkansas Nuclear 1,0.0,1998-12-28,Manual Reactor Trip. See event 35199. Design...,NaN,NaN
2,1999-01-03,4,Arkansas Nuclear 1,0.0,1998-12-28,Manual Reactor Trip. See event 35199. Design...,NaN,NaN
3,1999-01-04,4,Arkansas Nuclear 1,0.0,1998-12-28,Manual Reactor Trip. See event 35199. Design...,NaN,NaN
4,1999-01-05,4,Arkansas Nuclear 1,0.0,1998-12-28,Manual Reactor Trip. See event 35199. Design...,NaN,NaN


In [ ]:
# Returns a NumPy array of unique values (includes NaN if present)
df['unit_name'].unique()

<ArrowStringArray>
['Arkansas Nuclear 1', 'Arkansas Nuclear 2',    'Beaver Valley 1',
    'Beaver Valley 2',        'Braidwood 1',        'Braidwood 2',
     'Browns Ferry 1',     'Browns Ferry 2',     'Browns Ferry 3',
        'Brunswick 1',
 ...
     'Turkey Point 4',     'Vermont Yankee',           'Vogtle 1',
           'Vogtle 2',           'Vogtle 3',           'Vogtle 4',
        'Waterford 3',        'Watts Bar 1',        'Watts Bar 2',
       'Wolf Creek 1']
Length: 110, dtype: str

There are 110 reactors in total. Checking all columns' to see if there are duplicate values

In [6]:
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["unit_name", "date"])

print("Date range:", df["date"].min(), "to", df["date"].max())
print("Number of reactors:", df["unit_name"].nunique())
print("Duplicate reactor-date rows:", df.duplicated(["unit_name", "date"]).sum())

df.isna().sum()

Date range: 1999-01-01 00:00:00 to 2026-09-07 00:00:00
Number of reactors: 110
Duplicate reactor-date rows: 0


date                    0
region                  0
unit_name               0
power_pct               0
down_raw           925764
reason_comment     789640
change_raw         895544
scrams_raw        1015801
dtype: int64

Checking how much data is available for each reactor. Some reactors don't have data on certain days.

In [7]:
coverage = df.groupby("unit_name").agg(
    first_date=("date", "min"),
    last_date=("date", "max"),
    observations=("date", "nunique"),
)

coverage["expected_days"] = (
    coverage["last_date"] - coverage["first_date"]
).dt.days + 1

coverage["missing_days"] = (
    coverage["expected_days"] - coverage["observations"]
)

coverage.sort_values("missing_days", ascending=False)

,first_date,last_date,observations,expected_days,missing_days
unit_name,,,,,
Palisades,1999-01-01,2026-09-07,8925,10112,1187
LaSalle 2,2013-06-27,2026-09-07,3654,4821,1167
LaSalle 1,2013-06-27,2026-09-07,3654,4821,1167
La Salle 2,1999-01-01,2016-09-26,6450,6479,29
La Salle 1,1999-01-01,2016-09-26,6450,6479,29
...,...,...,...,...,...
Crystal River 3,1999-01-01,2013-06-07,5270,5272,2
Kewaunee,1999-01-01,2013-06-07,5270,5272,2
River Bend Station 1,2019-03-23,2026-09-07,2726,2726,0


Not all reactors have the same amount of data. Some came online recently, some were decommissioned.